In [1]:

import dotenv
import sys
print(sys.path)
import readline
# from openai import AsyncOpenAI
# client = AsyncOpenAI()
import os
import json

['/Library/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/jaslavie/Library/Python/3.11/lib/python/site-packages', '/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages']


In [2]:
# Load all texts from text folder
texts = []
text_dir = "../data/processed"
for file in os.listdir(text_dir):
    with open(os.path.join(text_dir, file), "r") as f:
        texts.append(f.read())

print(texts[0])

{
  "messages": [
    {
      "role": "assistant",
      "content": "9-1-1, what's your emergency?"
    },
    {
      "role": "user",
      "content": "Yeah, my father is stuck up in the attic. He's 79 years old."
    },
    {
      "role": "assistant",
      "content": "Why is he in the attic?"
    },
    {
      "role": "user",
      "content": "He's trying to fix something, but it's really hot up there. He went up about 50 feet. It's just me and my mother here, and we can't get him to come down. Can you please send someone to help?"
    },
    {
      "role": "assistant",
      "content": "Okay, but this does not sound like a life-threatening emergency. If he wants to be in the attic, the police can't make him come down. Is he capable of getting down himself?"
    },
    {
      "role": "user",
      "content": "It's hot up there, and I'm worried if he gets stuck, it could become dangerous. I'm just trying to prevent that."
    },
    {
      "role": "assistant",
      "content": "

In [3]:
system_message = """
I have a raw transcript of a 911 call with some words and sentences mistranscribed. I want to turn this into an OpenAI-compatible messages JSON array so that I can use this to fine tune an LLM on being a 911 operator. Clean up the transcript slightly, rewriting (but not ommitting) lines that make sense for the operator and caller to say. Use the "assistant" role for operator and "user" role for caller. Give me the messages array.

The messages array should be formatted like this:

{
    "messages": [
        {
        
            "role": "assistant",
            "content": "9-1-1, what's your emergency?"
        },
        {
            "location": "<location>",
            "name": "<name>",
            "role": "user",
            "emergency_type": "<emergency_type>",
            "content": "<caller message>"
        },
        ...
    ]
}

The output should always start with "9-1-1, what's your emergency?".

The user will input a transcript where each new line may have been said by the operator or the caller.
"""
async def process_text(text):
    result = await client.chat.completions.create(
        model="gpt-4o",
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": system_message}, # system message
            {"role": "user", "content": text} # raw transcript
        ]
    )
    return result.choices[0].message.content


In [4]:
import asyncio
from tqdm import tqdm
import json
import os
processed_texts_saved = []
async def process_batch(batch):
    tasks = [process_text(text) for text in batch]
    return await asyncio.gather(*tasks)

async def process_all_texts():
    global processed_texts_saved
    batch_size = 100
    delay = 30  # seconds
    processed_texts = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        results = await process_batch(batch)
        
        # Store processed results
        processed_texts.extend(results)
        
        # Save processed results to a file after each batch
        with open('processed_texts.json', 'w') as f:
            json.dump(processed_texts, f)
        
        if i + batch_size < len(texts):
            print(f"Waiting {delay} seconds before next batch...")
            await asyncio.sleep(delay)
            
    processed_texts_saved = processed_texts

    # After all batches are processed, save to individual files
    if not os.path.exists('processed'):
        os.makedirs('processed')
    
    for idx, processed_text in enumerate(processed_texts):
        try:
            with open(f'processed/text_{idx}.json', 'w') as f:
                json.dump(json.loads(processed_text), f, indent=2)
        except Exception as e:
            print(f"Error processing text {idx}: {e}")

# Run the async function
await process_all_texts()


ModuleNotFoundError: No module named 'tqdm'